# 39. Domain Lexicon v1.1 → v1.2

`spec.md` §8 의 미해결 2건을 닫는다.

- **6번** SCENE 표현(`빨래`·`이불`·`호텔`·`휴양지`·`비 오는 숲`)을 유지할 것인가
- **2번** `LLM` 등급 7행을 사람이 검토해 `TEAM` 으로 올릴 것인가

노트북 31 이 v1 → v1.1 을 만든 방식을 그대로 따른다. **v1.1 을 덮어쓰지 않는다.**

## 0. 실행 조건과 한계

- **API 호출 0회.** 기존 측정(노트북 30·34)과 사전만 읽는다
- **설문 155건을 쓰지 않는다.** `spec.md` §7.2 의 holdout 이고,
  그것으로 사전을 고치면 holdout 이 아니게 된다
- 기여도 근거는 **합성 평가셋 600건**에서 왔다 (holdout 아님)
- **`status` 를 건드리지 않는다.** `candidate` → `active` 는 사전 전체의 의미 판정이라 별건
- 삭제·교체를 하지 않는다. N5 의 *"강등하되 교체하지 않는다"* 를 이어간다

In [1]:
import hashlib
import json
import pathlib

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

LIFT_THRESHOLD = 1.4      # N5 가 통제군 최대 1.38 에서 정한 문턱. 그대로 쓴다
MIN_CORE = 2              # spec §4.3 단독 매핑 금지
STAMP = "[39] "           # 근거 칸에 붙일 출처 표시

print(f"REPORT_ONLY: {REPORT_ONLY} / 문턱: {LIFT_THRESHOLD} (N5 와 동일)")

REPORT_ONLY: False / 문턱: 1.4 (N5 와 동일)


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATHS = {
    "lexicon_v11": KNOWLEDGE_DIR / "domain_lexicon_v1_1.csv",
    "crosscheck": OUTPUT_DIR / "30_lexicon_crosscheck.csv",
}
OUTPUT_PATHS = {
    "lexicon_v12": KNOWLEDGE_DIR / "domain_lexicon_v1_2.csv",
    "changelog": OUTPUT_DIR / "39_lexicon_v1_2_changelog.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {p.resolve() for p in INPUT_PATHS.values()} | {
    (KNOWLEDGE_DIR / "domain_lexicon_v1.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "data" / "survey" / "processed"
     / "survey_nlp_queries_candidates.csv").resolve(),
    (PROJECT_ROOT / "perfumes.csv").resolve(),
    (PROJECT_ROOT / "perfumes.jsonl").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
lexicon_v11,9e0a1365437a27a0
crosscheck,41865413e557214f


## 2. 사전 등록 — 데이터를 고치기 전에 기준을 고정한다

사용자 승인 2026-09-12. **결과를 보고 기준을 바꾸지 않는다.**

In [3]:
DECISION = {
    "§8 6번 판정": "B — 배수 기준으로 가른다. N5 의 문턱 1.4 를 그대로 적용한다",
    "6번-1 지지": "배수가 문턱을 넘고 순환이 아니면 유지하고 근거를 기록한다",
    "6번-2 판정 불가": "표본 부족으로 배수를 못 낸 행은 **강등하지 않는다.** "
                       "측정 불가는 반증이 아니다 (N5 근거 1)",
    "6번-3 구조 변경": "이 판정으로 target_field·required 를 바꾸지 않는다. "
                       "v1.1 이 이미 문턱을 적용했다",
    "§8 2번 판정": "N4 배수 판정을 근거로 사람이 검토한다",
    "2번-1 지지": "LLM → TEAM 승격",
    "2번-2 약함": "LLM 등급 유지. 지우지도 올리지도 않는다",
    "2번-3 status": "candidate 그대로. active 승격은 별건",
    "공통": "삭제·교체 없음. 행 수 44 유지",
}

# 측정 기록 11 — 노트북 30 의 min_support=30 에 가려진 3건을 원자료로 재계산한 값.
# N4 판정표의 NaN 을 이 값으로 해석한다. 새로 계산하지 않고 기록된 수치를 인용한다.
REMEASURED = {
    ("kr.drift.musk", "soapy"): (2.15, "문턱 통과 — min_support 30 에 가려졌던 신호"),
    ("kr.scene.bedding", "soapy"): (None, "표본 부족 — 해당군 112개 중 보유 2개. 배수가 의미 없다"),
    ("kr.scene.hotel", "soapy"): (1.29, "문턱 미달"),
}
display(pd.Series(DECISION, name="사전 등록").to_frame())

,사전 등록
§8 6번 판정,B — 배수 기준으로 가른다. N5 의 문턱 1.4 를 그대로 적용한다
6번-1 지지,배수가 문턱을 넘고 순환이 아니면 유지하고 근거를 기록한다
6번-2 판정 불가,표본 부족으로 배수를 못 낸 행은 **강등하지 않는다.** 측정 불가는 반증이 아니다 (N5 근거 1)
6번-3 구조 변경,이 판정으로 target_field·required 를 바꾸지 않는다. v1.1 이 이미 문턱을 적용했다
§8 2번 판정,N4 배수 판정을 근거로 사람이 검토한다
2번-1 지지,LLM → TEAM 승격
2번-2 약함,LLM 등급 유지. 지우지도 올리지도 않는다
2번-3 status,candidate 그대로. active 승격은 별건
공통,삭제·교체 없음. 행 수 44 유지


## 3. 입력 — v1.1 과 N4 판정표

In [4]:
lex = pd.read_csv(INPUT_PATHS["lexicon_v11"], keep_default_na=False, dtype=str)
cc = pd.read_csv(INPUT_PATHS["crosscheck"])
print(f"사전 v1.1 {len(lex)}행 {len(lex.columns)}컬럼 / N4 판정표 {len(cc)}행")

verdict = {(r["entry_id"], r["accord"]): r for r in cc.to_dict("records")}
SCENE = sorted(set(lex[lex.entry_id.str.startswith("kr.scene.")]["expression"]))
print(f"SCENE 표현 {len(SCENE)}개: {SCENE}")
print(f"LLM 등급 {len(lex[lex.evidence_tier=='LLM'])}행")

사전 v1.1 44행 16컬럼 / N4 판정표 28행
SCENE 표현 5개: ['비 오는 숲', '빨래', '이불', '호텔', '휴양지']
LLM 등급 7행


## 4. §8 6번 — SCENE 판정

v1.1 이 이미 문턱을 적용했으므로 **구조는 바뀌지 않는다.** 판정 근거를 행에 기록한다.

In [5]:
def verdict_of(entry_id, accord):
    """N4 판정표에서 해당 행을 찾는다. (판정, 배수, 순위) 또는 (None, None, None)."""
    r = verdict.get((entry_id, accord))
    if r is None:
        return None, None, None
    lift = r["배수"]
    if pd.isna(lift) and (entry_id, accord) in REMEASURED:
        re_lift, re_note = REMEASURED[(entry_id, accord)]
        return f"재계산(측정기록11) — {re_note}", re_lift, None
    return r["판정"], (None if pd.isna(lift) else float(lift)), r["배수 순위"]


scene_log = []
for i, r in lex.iterrows():
    if not r["entry_id"].startswith("kr.scene.") or r["candidate_type"] != "ACCORD":
        continue
    v, lift, rank = verdict_of(r["entry_id"], r["candidate_name"])
    if v is None:
        note, act = "N4 판정표에 없음", "변경 없음"
    elif "순환" in str(v):
        note, act = "검색어가 accord 이름과 같아 신호로 읽지 않음", "변경 없음"
    elif str(v).startswith("재계산"):
        if lift is None:
            note, act = v + ". 측정 불가는 반증이 아니므로 강등하지 않는다", "유지 · 근거 한계 기록"
        elif lift >= LIFT_THRESHOLD:
            note, act = f"{v} (배수 {lift:.2f})", "유지"
        else:
            note, act = f"{v} (배수 {lift:.2f})", "이미 v1.1 에서 처리됨"
    elif lift is None:
        note = ("표본 부족으로 배수를 내지 못함. 측정 불가는 반증이 아니므로 강등하지 않는다")
        act = "유지 · 근거 한계 기록"
    elif lift >= LIFT_THRESHOLD:
        note, act = f"배수 {lift:.2f}" + (f" · {rank:.0f}위" if rank is not None else "") + f"로 문턱 {LIFT_THRESHOLD} 통과", "유지"
    else:
        note, act = f"배수 {lift:.2f}" + (f" · {rank:.0f}위" if rank is not None else "") + "로 문턱 미달", "이미 v1.1 에서 처리됨"
    scene_log.append({"표현": r["expression"], "accord": r["candidate_name"],
                      "required": r["required"], "target_field": r["target_field"],
                      "판정": v, "배수": lift, "조치": act, "근거": note})
    lex.at[i, "rationale"] = (str(r["rationale"]).rstrip() + " " + STAMP + note).strip()

scene_df = pd.DataFrame(scene_log)
display(scene_df[["표현", "accord", "required", "판정", "배수", "조치"]])
print("\nSCENE 표현별 결론")
for e in SCENE:
    g = lex[(lex.expression == e) & (lex.candidate_type == "ACCORD")]
    core = g[g.required == "core"]
    print(f"  {e:10s} core {len(core)}개 · target={g.target_field.iloc[0]} → "
          f"{'유지' if g.target_field.iloc[0] != 'NO_MAPPING' else 'NO_MAPPING 유지'}")

,표현,accord,required,판정,배수,조치
0,빨래,soapy,core,지지,4.497194,유지
1,빨래,fresh,core,순환 — 신호로 읽지 않음,1.765685,변경 없음
2,이불,powdery,core,지지,1.567686,유지
3,이불,soapy,core,재계산(측정기록11) — 표본 부족 — 해당군 112개 중 보유 2개. 배수가 의미 없다,NaN,유지 · 근거 한계 기록
4,이불,musky,optional,약함,1.454012,유지
5,비 오는 숲,mossy,core,지지,3.628034,유지
6,비 오는 숲,earthy,core,지지,3.036363,유지
7,비 오는 숲,green,optional,지지,1.999919,유지
8,휴양지,tropical,core,순환 — 신호로 읽지 않음,5.735430,변경 없음
9,휴양지,coconut,core,지지,5.757615,유지



SCENE 표현별 결론
  비 오는 숲     core 2개 · target=STAGE2_BRIDGE → 유지
  빨래         core 2개 · target=STAGE2_BRIDGE → 유지
  이불         core 2개 · target=STAGE2_BRIDGE → 유지
  호텔         core 0개 · target=NO_MAPPING → NO_MAPPING 유지
  휴양지        core 2개 · target=STAGE2_BRIDGE → 유지


## 5. §8 2번 — `LLM` 등급 7행 검토

In [6]:
llm_log = []
for i, r in lex.iterrows():
    if r["evidence_tier"] != "LLM":
        continue
    v, lift, rank = verdict_of(r["entry_id"], r["candidate_name"])
    if v == "지지":
        new_tier = "TEAM"
        note = (f"N4 배수 {lift:.2f}" + (f" · {rank:.0f}위" if rank is not None else "") + "로 지지. 사람 검토로 TEAM 승격 "
                "(판정 기준 2번-1)")
    else:
        new_tier = "LLM"
        note = (f"N4 판정 '{v}'"
                + (f" (배수 {lift:.2f}" + (f" · {rank:.0f}위)" if rank is not None else ")") if lift is not None else "")
                + ". 승격도 강등도 하지 않는다 (판정 기준 2번-2)")
    llm_log.append({"표현": r["expression"], "accord": r["candidate_name"],
                    "required": r["required"], "판정": v, "배수": lift,
                    "등급": f"LLM → {new_tier}" if new_tier != "LLM" else "LLM 유지"})
    lex.at[i, "evidence_tier"] = new_tier
    if new_tier == "TEAM":
        lex.at[i, "rationale"] = (str(r["rationale"]).rstrip() + " " + STAMP + note).strip()
    else:
        lex.at[i, "rationale"] = (str(r["rationale"]).rstrip() + " " + STAMP + note).strip()

llm_df = pd.DataFrame(llm_log)
display(llm_df)
print(f"\n승격 {sum(1 for r in llm_log if 'TEAM' in r['등급'])}행 / "
      f"유지 {sum(1 for r in llm_log if r['등급'] == 'LLM 유지')}행")

,표현,accord,required,판정,배수,등급
0,머스크,fresh,optional,약함,1.268350,LLM 유지
1,깨끗한,fresh,core,지지,1.610654,LLM → TEAM
2,이불,musky,optional,약함,1.454012,LLM 유지
3,비 오는 숲,green,optional,지지,1.999919,LLM → TEAM
4,휴양지,coconut,core,지지,5.757615,LLM → TEAM
5,호텔,white floral,optional,약함,1.099134,LLM 유지
6,달달,caramel,core,지지,1.614265,LLM → TEAM



승격 4행 / 유지 3행


## 6. 검증 — 바뀌면 안 되는 칸이 안 바뀌었는가

In [7]:
before = pd.read_csv(INPUT_PATHS["lexicon_v11"], keep_default_na=False, dtype=str)
problems = []

if len(lex) != len(before):
    problems.append(f"행 수가 {len(before)} → {len(lex)} 로 바뀌었다")
if list(lex.columns) != list(before.columns):
    problems.append("컬럼 구성이 바뀌었다")

FROZEN = ["entry_id", "expression", "aliases", "expression_type", "target_field",
          "candidate_type", "candidate_name", "rank", "required", "match_condition",
          "corpus_support", "standardness", "status", "source_query_ids"]
for c in FROZEN:
    n = int((lex[c] != before[c]).sum())
    if n:
        problems.append(f"고정 칸 {c} 가 {n}행 바뀌었다")

tier_changed = int((lex["evidence_tier"] != before["evidence_tier"]).sum())
if tier_changed != 4:
    problems.append(f"evidence_tier 변경이 {tier_changed}행 (기대 4행)")
# 매핑 행(ACCORD)만 candidate 여야 한다. FIELD·NO_MAPPING 은 v1 부터 active 다.
acc_rows = lex[lex.candidate_type == "ACCORD"]
if not (acc_rows["status"] == "candidate").all():
    problems.append("ACCORD 매핑 행에 candidate 가 아닌 status 가 있다")
if int((lex["status"] != before["status"]).sum()):
    problems.append("status 가 바뀐 행이 있다")

# core 최소 개수 (spec §4.3)
acc = lex[(lex.candidate_type == "ACCORD") & (lex.target_field != "NO_MAPPING")]
for (expr, cond), g in acc.groupby(["expression", "match_condition"]):
    n_core = int((g.required == "core").sum())
    if n_core < MIN_CORE:
        problems.append(f"{expr}({cond or '조건없음'}) 의 core 가 {n_core}개")

print("검증 통과" if not problems else "검증 실패")
for p in problems:
    print(" -", p)
print(f"\nevidence_tier 변경 {tier_changed}행 / rationale 변경 "
      f"{int((lex['rationale'] != before['rationale']).sum())}행")
display(pd.crosstab(lex["evidence_tier"], lex["candidate_type"], margins=True))

검증 통과

evidence_tier 변경 4행 / rationale 변경 15행


candidate_type,,ACCORD,FIELD,All
evidence_tier,,,,
LLM,0,3,0,3
TEAM,4,23,0,27
VERIFIED,4,2,8,14
All,8,28,8,44


## 7. 저장

In [8]:
changed = lex["evidence_tier"] != before["evidence_tier"]
lines = [
    "# 39. Domain Lexicon v1.1 → v1.2 변경 기록",
    "",
    "사전 등록: 노트북 39 cell 6 (사용자 승인 2026-09-12)",
    "근거: `DECISIONS.md` N4 배수 판정 · `analysis_outputs/30_lexicon_crosscheck.csv`",
    "",
    "## 결론",
    "",
    f"- **§8 6번 SCENE** — 판정 B(배수 기준). **구조 변경 0행.** "
    f"v1.1 이 이미 문턱 {LIFT_THRESHOLD} 를 적용했으므로 근거만 기록했다",
    f"- **§8 2번 LLM 등급** — 지지 {int(changed.sum())}행을 `TEAM` 으로 승격, "
    f"약함 {int((lex.evidence_tier == 'LLM').sum())}행은 유지",
    "- `status` 는 전부 `candidate` 그대로. 삭제·교체 없음. 44행 유지",
    "",
    "## evidence_tier 가 바뀐 행",
    "",
    "| 표현 | accord | required | N4 판정 | 배수 | 등급 |",
    "|---|---|---|---|---:|---|",
]
for r in llm_log:
    if "TEAM" in r["등급"]:
        lines.append(f"| {r['표현']} | `{r['accord']}` | {r['required']} | {r['판정']} | "
                     f"{r['배수']:.2f} | {r['등급']} |")
lines += ["", "## 등급을 올리지 않은 행", "",
          "| 표현 | accord | N4 판정 | 배수 |", "|---|---|---|---:|"]
for r in llm_log:
    if r["등급"] == "LLM 유지":
        lift = f"{r['배수']:.2f}" if r["배수"] is not None else "—"
        lines.append(f"| {r['표현']} | `{r['accord']}` | {r['판정']} | {lift} |")
lines += ["", "## SCENE 판정", "",
          "| 표현 | accord | 판정 | 배수 | 조치 |", "|---|---|---|---:|---|"]
for r in scene_log:
    lift = f"{r['배수']:.2f}" if r["배수"] is not None else "—"
    lines.append(f"| {r['표현']} | `{r['accord']}` | {r['판정']} | {lift} | {r['조치']} |")
lines += [
    "", "## 한계", "",
    "- **`이불` 의 배수 근거는 신뢰할 수 없다.** 해당군이 112개뿐이고, 측정 기록 10이",
    "  `bedding|cotton|blanket` 검색이 `cotton candy` 를 잡은 오염을 기록했다.",
    "  그래서 `soapy` 를 강등도 승격도 하지 않았다",
    "- **평가셋에서 SCENE 이 걸린 횟수는 600문장 중 23건**이다. 유지 결정의 이득이 크지 않다",
    "- 설문 155건은 holdout 이라 이 판정에 쓰지 않았다",
    "",
]
write_output(OUTPUT_PATHS["lexicon_v12"],
             lambda p: lex.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["changelog"],
             lambda p: p.write_text("\n".join(lines), encoding="utf-8"))

저장: data\scent_knowledge\domain_lexicon_v1_2.csv
저장: analysis_outputs\39_lexicon_v1_2_changelog.md


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/analysis_outputs/39_lexicon_v1_2_changelog.md')

## 8. 가드 검증

In [9]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
ch = [k for k in after if after[k] != input_hashes_before[k]]
if ch:
    raise RuntimeError(f"입력 파일이 변경됐다: {ch}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
v1 = KNOWLEDGE_DIR / "domain_lexicon_v1.csv"
if v1.is_file():
    print("v1 보존본 존재 확인 / sha256", sha256_file(v1)[:16])
print()
print("남은 것 — spec.md §8 의 6번·2번 행과 §10 개정 이력, DECISIONS.md 기록")

입력 해시 불변 확인: lexicon_v11, crosscheck
v1 보존본 존재 확인 / sha256 936808a9b3c32c2b

남은 것 — spec.md §8 의 6번·2번 행과 §10 개정 이력, DECISIONS.md 기록
